# 🎵 Music Genre Classification — AST Fine-Tuning
### Target: 0.80+ Macro F1
**Strategy:** Two-phase AST fine-tuning + Strong Augmentation + TTA

---
### ⚙️ Key Parameters (Change these to tune score)
| Parameter | Default | Higher = ? |
|-----------|---------|------------|
| `DURATION` | 20s | More context, slower |
| `TRAIN_SIZE` | 5000 | More diversity |
| `EPOCHS_P2` | 5 | More fine-tuning |
| `LR_P2` | 2e-5 | Careful! Too high destroys model |
| `N_TTA` | 5 | Better inference, slower |
| `NOISE_PROB` | 0.7 | More robustness |

In [1]:
# ── CELL 1: Install libraries ──────────────────────────
!pip install -q transformers librosa torchmetrics wandb torchaudio

In [2]:
# ── CELL 2: Imports ────────────────────────────────────
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import librosa
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    get_cosine_schedule_with_warmup
)
from torchmetrics.classification import MulticlassF1Score
warnings.filterwarnings('ignore')
print('All imports done!')

All imports done!


In [3]:
# ── CELL 3: Seed for reproducibility ──────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print('Seed set!')

Seed set!


In [4]:
# ── CELL 4: CONFIG — Change these to tune your score! ──
# -------------------------------------------------------
# 🔧 MAIN TUNING KNOBS:
#   DURATION   ↑ → more audio context (try 25 or 30)
#   TRAIN_SIZE ↑ → more augmented samples (try 6000)
#   EPOCHS_P2  ↑ → more fine-tuning (try 6 or 7)
#   LR_P2      ↓ → safer fine-tune (try 1e-5)
#   N_TTA      ↑ → better inference (try 7)
#   NOISE_PROB ↑ → more noise robustness (try 0.8)
# -------------------------------------------------------

SAMPLE_RATE  = 16000
DURATION     = 20                        # seconds of audio per sample
MAX_LENGTH   = SAMPLE_RATE * DURATION

BATCH_SIZE   = 8                         # keep small for GPU memory
GRAD_ACCUM   = 2                         # effective batch = 6x2 = 12
# More workers are safe now: __getitem__ only does I/O + mixing (no spectrogram)
NUM_WORKERS  = 4
PREFETCH     = 4                         # prefetch batches per worker

EPOCHS_P1    = 2                         # phase 1: freeze base, train head
EPOCHS_P2    = 8                         # phase 2: full fine-tune
LR_P1        = 3e-4                      # phase 1 lr (higher ok, base frozen)
LR_P2        = 2e-5                      # phase 2 lr (KEEP SMALL!)
LR_HEAD_MULT = 10                        # classifier gets LR_P2 * 10
WEIGHT_DECAY = 0.01

TRAIN_SIZE   = 4000                      # synthetic training samples
NOISE_PROB   = 0.7                       # probability of adding noise
NOISE_LEVEL  = (0.03, 0.20)             # min/max noise volume
# tempo_augment removed — replaced by SpecAugment on GPU (faster & proven for AST)
PATIENCE_P1  = 2                         # early stopping patience phase 1
PATIENCE_P2  = 3                         # early stopping patience phase 2

N_TTA        = 5                         # test time augmentation crops

# SpecAugment params (applied on GPU in training loop)
TIME_MASK_PARAM = 192                    # max consecutive time frames to mask
FREQ_MASK_PARAM = 48                     # max consecutive mel bins to mask

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device     : {DEVICE}')
print(f'Duration   : {DURATION}s')
print(f'Train size : {TRAIN_SIZE}')
print(f'Epochs P1  : {EPOCHS_P1}  |  Epochs P2: {EPOCHS_P2}')
print(f'LR P1      : {LR_P1}  |  LR P2: {LR_P2}')
print(f'TTA crops  : {N_TTA}')
print(f'Num workers: {NUM_WORKERS}  |  Prefetch: {PREFETCH}')

Device     : cuda
Duration   : 20s
Train size : 2500
Epochs P1  : 3  |  Epochs P2: 5
LR P1      : 0.0003  |  LR P2: 2e-05
TTA crops  : 5
Num workers: 4  |  Prefetch: 4


In [5]:
# ── CELL 5: Paths & Labels ─────────────────────────────
BASE_PATH      = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES_PATH    = os.path.join(BASE_PATH, 'genres_stems')
ESC_PATH       = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
REQUIRED_STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

GENRES   = ['blues','classical','country','disco','hiphop',
            'jazz','metal','pop','reggae','rock']
label2id = {g: i for i, g in enumerate(GENRES)}
id2label = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

print('Paths set!')
print('label2id:', label2id)

Paths set!
label2id: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [6]:
# ── CELL 6: Build Song Index ───────────────────────────
def build_song_index():
    index = {}
    for genre in GENRES:
        genre_path = os.path.join(GENRES_PATH, genre)
        valid = []
        for folder in sorted(os.listdir(genre_path)):
            folder_path = os.path.join(genre_path, folder)
            if not os.path.isdir(folder_path):
                continue
            # only add if ALL 4 stems exist
            if all(os.path.exists(os.path.join(folder_path, s))
                   for s in REQUIRED_STEMS):
                valid.append(folder_path)
        index[genre] = valid
        print(f'  {genre}: {len(valid)} songs')
    return index

song_index = build_song_index()

# collect noise files
noise_files = []
for root, _, files in os.walk(ESC_PATH):
    for f in files:
        if f.endswith('.wav'):
            noise_files.append(os.path.join(root, f))

print(f'\nNoise files: {len(noise_files)}')

  blues: 100 songs
  classical: 100 songs
  country: 100 songs
  disco: 100 songs
  hiphop: 100 songs
  jazz: 100 songs
  metal: 100 songs
  pop: 100 songs
  reggae: 100 songs
  rock: 100 songs

Noise files: 2000


In [7]:
# ── CELL 7: Train / Val Split ──────────────────────────
train_index = {}
val_index   = {}

for genre in GENRES:
    songs = song_index[genre][:]
    random.shuffle(songs)
    split = int(0.85 * len(songs))          # 85% train, 15% val
    train_index[genre] = songs[:split]
    val_index[genre]   = songs[split:]
    print(f'  {genre} → Train: {len(train_index[genre])}, Val: {len(val_index[genre])}')

print('\nTrain/Val split done!')

  blues → Train: 85, Val: 15
  classical → Train: 85, Val: 15
  country → Train: 85, Val: 15
  disco → Train: 85, Val: 15
  hiphop → Train: 85, Val: 15
  jazz → Train: 85, Val: 15
  metal → Train: 85, Val: 15
  pop → Train: 85, Val: 15
  reggae → Train: 85, Val: 15
  rock → Train: 85, Val: 15

Train/Val split done!


In [8]:
# ── CELL 8: Audio Helper Functions ────────────────────
# torchaudio.load is ~3-5x faster than librosa.load and
# avoids resampling overhead for files already at 16 kHz.

def load_audio(path):
    """Load mono audio at SAMPLE_RATE using torchaudio (faster than librosa)."""
    waveform, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    if waveform.shape[0] > 1:            # stereo → mono
        waveform = waveform.mean(0, keepdim=True)
    return waveform.squeeze(0).numpy().astype(np.float32)

def normalize(audio):
    return audio / (np.max(np.abs(audio)) + 1e-6)

def crop_random(audio):
    """Random crop — used during training (augmentation)"""
    if len(audio) >= MAX_LENGTH:
        start = random.randint(0, len(audio) - MAX_LENGTH)
        return audio[start : start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

def crop_center(audio):
    """Centre crop — used during validation (reproducible)"""
    if len(audio) >= MAX_LENGTH:
        start = (len(audio) - MAX_LENGTH) // 2
        return audio[start : start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

def random_gain(audio):
    """Random volume change"""
    return audio * random.uniform(0.7, 1.3)

# NOTE: tempo_augment (librosa.effects.time_stretch) has been REMOVED.
# Replaced by GPU SpecAugment (time-masking) in GPUAudioTransform.
# time_stretch was the single slowest CPU op per sample (~50-200ms).

# noise_arrays is populated in the next cell (RAM pre-load)
def add_noise(audio):
    """Add pre-loaded environmental noise — no disk I/O at call time."""
    if random.random() < NOISE_PROB and len(noise_arrays) > 0:
        noise = random.choice(noise_arrays)   # already in RAM
        noise = crop_random(noise)
        level = random.uniform(*NOISE_LEVEL)
        audio = audio + level * noise
    return audio

print('Audio helpers ready!')

Audio helpers ready!


In [9]:
# ── CELL 8b: Pre-load ESC-50 Noise Files to RAM ────────
# WHY: add_noise() previously called librosa.load() on every __getitem__
# call — that's a random disk read inside every training sample.
# Loading all ESC-50 clips (~2000 × 5s) into RAM is ~200-400 MB and
# eliminates all noise-related disk I/O during training.

print('Pre-loading ESC-50 noise files to RAM...')
noise_arrays = []
for nf in tqdm(noise_files, desc='Loading noise'):
    try:
        noise_arrays.append(load_audio(nf))
    except Exception:
        pass

total_mb = sum(a.nbytes for a in noise_arrays) / 1e6
print(f'Loaded {len(noise_arrays)} noise clips into RAM ({total_mb:.0f} MB)')

Pre-loading ESC-50 noise files to RAM...


Loading noise: 100%|██████████| 2000/2000 [00:48<00:00, 40.99it/s]

Loaded 2000 noise clips into RAM (640 MB)


In [10]:
# ── CELL 9: Load AST Feature Extractor & Model ────────
# ASTFeatureExtractor is still used for TTA inference only.
# During training, spectrogram computation is done on GPU
# by GPUAudioTransform (defined in the next cell).

MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)
print('Feature extractor loaded (used for TTA only)!')

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels             = NUM_LABELS,
    id2label               = id2label,
    label2id               = label2id,
    ignore_mismatched_sizes = True
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'AST model loaded on {DEVICE}')
print(f'Total parameters: {total_params:,}')

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Feature extractor loaded (used for TTA only)!


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


AST model loaded on cuda
Total parameters: 86,196,490


In [18]:
# ── CELL 9b: GPU Audio Transform ───────────────────────
# WHY THIS MATTERS:
# The original __getitem__ ran ASTFeatureExtractor (CPU) on every sample,
# blocking the DataLoader workers and starving the GPU.
#
# GPUAudioTransform replicates the same log-mel spectrogram but runs as a
# batched GPU kernel — processing a full batch in one fused CUDA call.
#
# It also adds SpecAugment (time + frequency masking) on the GPU,
# replacing the slow librosa.time_stretch that ran per-sample on CPU.

class GPUAudioTransform(nn.Module):
    """
    Batched GPU pipeline: raw waveform → AST-compatible log-mel spectrogram.

    Matches ASTFeatureExtractor normalisation so pretrained weights are valid:
      normalised = (dB_spec - AST_MEAN) / (AST_STD * 2)

    When augment=True (training), applies SpecAugment after normalisation —
    this replaces the removed librosa.time_stretch CPU augmentation.
    """
    # Normalisation constants from ASTFeatureExtractor
    AST_MEAN = -4.2677393
    AST_STD  =  4.5689974

    def __init__(
        self,
        sample_rate     = SAMPLE_RATE,
        n_mels          = 128,
        n_fft           = 400,          # 25 ms window @ 16 kHz
        hop_length      = 313,          # 10 ms hop @ 16 kHz
        f_min           = 50.0,
        f_max           = 8000.0,
        time_mask_param = TIME_MASK_PARAM,
        freq_mask_param = FREQ_MASK_PARAM,
    ):
        super().__init__()
        # Batched mel spectrogram — runs on GPU as a single CUDA op
        self.mel_spec = T.MelSpectrogram(
            sample_rate = sample_rate,
            n_fft       = n_fft,
            hop_length  = hop_length,
            n_mels      = n_mels,
            f_min       = f_min,
            f_max       = f_max,
            power       = 2.0,
            center      = True,
        )
        self.amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=80)

        # SpecAugment — runs on GPU, replaces tempo_augment on CPU
        # Applied twice each for stronger regularisation
        self.time_mask_1 = T.TimeMasking(time_mask_param, iid_masks=True)
        self.time_mask_2 = T.TimeMasking(time_mask_param, iid_masks=True)
        self.freq_mask_1 = T.FrequencyMasking(freq_mask_param, iid_masks=True)
        self.freq_mask_2 = T.FrequencyMasking(freq_mask_param, iid_masks=True)

    def forward(self, waveforms: torch.Tensor, augment: bool = False) -> torch.Tensor:
        """
        Args:
            waveforms : (B, T)  float32 — raw audio already on GPU
            augment   : bool    — apply SpecAugment (True during training)
        Returns:
            spec      : (B, T_frames, n_mels)  — AST input_values format
        """
        spec = self.mel_spec(waveforms)          # (B, n_mels, T_frames)
        spec = self.amplitude_to_db(spec)         # (B, n_mels, T_frames)  dB scale
        # Normalise with AST constants
        spec = (spec - self.AST_MEAN) / (self.AST_STD * 2)

        if augment:
            spec = self.time_mask_1(spec)         # randomly zero time bands
            spec = self.time_mask_2(spec)
            spec = self.freq_mask_1(spec)         # randomly zero freq bands
            spec = self.freq_mask_2(spec)

        spec = spec.transpose(1, 2)               # (B, T_frames, n_mels) ← AST shape
        return spec


# Instantiate and move to GPU alongside the model
gpu_transform = GPUAudioTransform().to(DEVICE)
print('GPUAudioTransform ready on', DEVICE)

# Quick shape check
with torch.no_grad():
    dummy = torch.randn(2, MAX_LENGTH).to(DEVICE)
    out = gpu_transform(dummy, augment=True)
    print(f'Transform output shape: {out.shape}  (B, T_frames, n_mels)')

GPUAudioTransform ready on cuda
Transform output shape: torch.Size([2, 1023, 128])  (B, T_frames, n_mels)


In [19]:
# ── CELL 10: Training Dataset (Synthetic Mashups) ─────
# __getitem__ now returns a raw WAVEFORM tensor, not a spectrogram.
#
# All spectrogram + augmentation work has moved to GPUAudioTransform
# which runs on the GPU inside the training loop.
#
# Changes vs original:
#   ✗ Removed: ASTFeatureExtractor call (was the #1 CPU bottleneck)
#   ✗ Removed: librosa.time_stretch (was #2 CPU bottleneck, ~100ms/sample)
#   ✓ Added  : returns raw float32 waveform tensor
#   ✓ Added  : no feature_extractor arg needed

class TrainDataset(Dataset):
    """
    Generates TRAIN_SIZE synthetic mashups on the fly.
    Returns raw waveform — spectrogram is computed on GPU in the train loop.
    """
    def __init__(self, song_index, size=TRAIN_SIZE):
        self.song_index = song_index
        self.size       = size

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        while True:
            try:
                genre = random.choice(GENRES)
                songs = self.song_index[genre]
                if not songs:
                    continue

                # Cross-song mixing: each stem from a DIFFERENT song
                mixed = None
                for stem in REQUIRED_STEMS:
                    song_path = random.choice(songs)
                    audio     = load_audio(os.path.join(song_path, stem))
                    audio     = crop_random(audio)
                    audio     = random_gain(audio)
                    mixed = audio if mixed is None else mixed + audio

                mixed = normalize(mixed)
                mixed = add_noise(mixed)    # noise read from RAM — no disk I/O
                mixed = normalize(mixed)

                # Return raw waveform — spectrogram computed on GPU later
                return {
                    'waveform': torch.from_numpy(mixed),    # (T,) float32
                    'label'   : torch.tensor(label2id[genre], dtype=torch.long)
                }
            except Exception:
                continue

print('TrainDataset class defined!')

TrainDataset class defined!


In [20]:
# ── CELL 11: Validation Dataset (Fixed Crops) ─────────
# Same change: returns raw waveform, not spectrogram.
# Spectrogram computed on GPU in validate().

class ValDataset(Dataset):
    """
    Loads actual songs with centre crop — reproducible evaluation.
    Returns raw waveform tensor.
    """
    def __init__(self, song_index):
        self.samples = []
        for genre in GENRES:
            for song_path in song_index[genre]:
                self.samples.append((genre, song_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        genre, song_path = self.samples[idx]
        mixed = None
        for stem in REQUIRED_STEMS:
            audio = load_audio(os.path.join(song_path, stem))
            audio = crop_center(audio)
            mixed = audio if mixed is None else mixed + audio

        mixed = normalize(mixed)
        return {
            'waveform': torch.from_numpy(mixed),
            'label'   : torch.tensor(label2id[genre], dtype=torch.long)
        }

print('ValDataset class defined!')

ValDataset class defined!


In [21]:
# ── CELL 12: Create DataLoaders ───────────────────────
# Note: no feature_extractor argument needed any more

train_dataset = TrainDataset(train_index, size=TRAIN_SIZE)
val_dataset   = ValDataset(val_index)

train_loader = DataLoader(
    train_dataset,
    batch_size         = BATCH_SIZE,
    shuffle            = True,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = True,
    prefetch_factor    = PREFETCH,       # overlap I/O with GPU compute
)
val_loader = DataLoader(
    val_dataset,
    batch_size         = BATCH_SIZE,
    shuffle            = False,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = True,
    prefetch_factor    = PREFETCH,
)

print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

Train samples : 2500
Val samples   : 150
Train batches : 417
Val batches   : 25


In [22]:
# ── CELL 13: Loss + F1 Metric ─────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
f1_metric = MulticlassF1Score(
    num_classes = NUM_LABELS,
    average     = 'macro'
).to(DEVICE)

print('Loss: CrossEntropyLoss with label_smoothing=0.1')
print('Metric: Macro F1 Score')

Loss: CrossEntropyLoss with label_smoothing=0.1
Metric: Macro F1 Score


In [23]:
# ── CELL 14: train / validate Functions ───────────────
# Key change: gpu_transform() called right after .to(DEVICE).
# This converts raw waveforms → log-mel spectrograms on GPU
# with SpecAugment applied during training.

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    f1_metric.reset()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc='Training')):
        waveforms = batch['waveform'].to(DEVICE)   # (B, T)
        labels    = batch['label'].to(DEVICE)

        # ── GPU TRANSFORM ────────────────────────────────
        # Batched mel spectrogram + AST normalisation + SpecAugment
        # Everything runs in CUDA — replaces the CPU ASTFeatureExtractor
        # that previously ran inside DataLoader workers.
        input_values = gpu_transform(waveforms, augment=True)  # (B, T_frames, 128)

        outputs = model(input_values=input_values)
        loss    = criterion(outputs.logits, labels) / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM
        preds = torch.argmax(outputs.logits, dim=1)
        f1_metric.update(preds, labels)

    return total_loss / len(loader), f1_metric.compute().item()


def validate(model, loader):
    model.eval()
    f1_metric.reset()
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            waveforms = batch['waveform'].to(DEVICE)
            labels    = batch['label'].to(DEVICE)
            # augment=False for reproducible validation
            input_values = gpu_transform(waveforms, augment=False)
            outputs      = model(input_values=input_values)
            preds        = torch.argmax(outputs.logits, dim=1)
            f1_metric.update(preds, labels)
    return f1_metric.compute().item()


print('train_one_epoch and validate functions ready!')

train_one_epoch and validate functions ready!


In [24]:
# ── CELL 15: PHASE 1 — Freeze base, train classifier ──
# WHY: Pretrained weights are precious. If we fine-tune
# everything immediately with large LR, we destroy them.
# Phase 1 warms up the new 10-class head first.

# freeze all base model params
for param in model.base_model.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1: Only {trainable:,} params trainable (classifier head only)')

optimizer_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = LR_P1,
    weight_decay = WEIGHT_DECAY
)
total_steps_p1 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P1
scheduler_p1   = get_cosine_schedule_with_warmup(
    optimizer_p1,
    num_warmup_steps   = total_steps_p1 // 10,
    num_training_steps = total_steps_p1
)

best_p1_f1   = 0
patience_cnt = 0

print(f'\n{"="*50}')
print(f'PHASE 1 — Classifier Warmup ({EPOCHS_P1} epochs)')
print(f'{"="*50}\n')

for epoch in range(EPOCHS_P1):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer_p1, scheduler_p1
    )
    val_f1 = validate(model, val_loader)

    print(f'Epoch {epoch+1}/{EPOCHS_P1}')
    print(f'  Train Loss : {train_loss:.4f}')
    print(f'  Train F1   : {train_f1:.4f}')
    print(f'  Val F1     : {val_f1:.4f}')

    if val_f1 > best_p1_f1:
        best_p1_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/best_model_phase1.pth')
        patience_cnt = 0
        print(f'  ✓ Saved best Phase 1 model (val F1: {best_p1_f1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE_P1:
            print('  Early stopping Phase 1.')
            break

print(f'\nPhase 1 done! Best val F1: {best_p1_f1:.4f}')

Phase 1: Only 9,226 params trainable (classifier head only)

PHASE 1 — Classifier Warmup (3 epochs)



Validation: 100%|██████████| 25/25 [00:23<00:00,  1.05it/s]


Epoch 1/3
  Train Loss : 2.1092
  Train F1   : 0.2799
  Val F1     : 0.4127
  ✓ Saved best Phase 1 model (val F1: 0.4127)


Validation: 100%|██████████| 25/25 [00:15<00:00,  1.57it/s]


Epoch 2/3
  Train Loss : 1.7137
  Train F1   : 0.5138
  Val F1     : 0.5874
  ✓ Saved best Phase 1 model (val F1: 0.5874)


Validation: 100%|██████████| 25/25 [00:14<00:00,  1.67it/s]

Epoch 3/3
  Train Loss : 1.6157
  Train F1   : 0.5746
  Val F1     : 0.5823

Phase 1 done! Best val F1: 0.5874


In [25]:
# ── CELL 16: PHASE 2 — Full fine-tune all layers ──────
# WHY: Now that classifier head is stable, we unfreeze
# everything and fine-tune at a TINY learning rate.
# Layer-wise LR: base gets LR_P2, classifier gets 10x more.

# load best phase 1 checkpoint
model.load_state_dict(torch.load('/kaggle/working/best_model_phase1.pth'))

# unfreeze ALL layers
for param in model.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
print(f'All {total_params:,} params unfrozen!')

# layer-wise learning rates
optimizer_p2 = torch.optim.AdamW([
    {'params': model.base_model.parameters(),
     'lr'    : LR_P2},                          # tiny LR for pretrained
    {'params': model.classifier.parameters(),
     'lr'    : LR_P2 * LR_HEAD_MULT},           # 10x for classifier
], weight_decay=WEIGHT_DECAY)

total_steps_p2 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P2
scheduler_p2   = get_cosine_schedule_with_warmup(
    optimizer_p2,
    num_warmup_steps   = total_steps_p2 // 10,
    num_training_steps = total_steps_p2
)

best_p2_f1   = 0
patience_cnt = 0

print(f'\n{"="*50}')
print(f'PHASE 2 — Full Fine-Tune ({EPOCHS_P2} epochs)')
print(f'Base LR: {LR_P2}  |  Head LR: {LR_P2*LR_HEAD_MULT}')
print(f'{"="*50}\n')

for epoch in range(EPOCHS_P2):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer_p2, scheduler_p2
    )
    val_f1 = validate(model, val_loader)

    print(f'Epoch {epoch+1}/{EPOCHS_P2}')
    print(f'  Train Loss : {train_loss:.4f}')
    print(f'  Train F1   : {train_f1:.4f}')
    print(f'  Val F1     : {val_f1:.4f}')

    if val_f1 > best_p2_f1:
        best_p2_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/best_model_phase2.pth')
        patience_cnt = 0
        print(f'  ✓ Saved best Phase 2 model (val F1: {best_p2_f1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE_P2:
            print('  Early stopping Phase 2.')
            break

print(f'\nPhase 2 done! Best val F1: {best_p2_f1:.4f}')

All 86,196,490 params unfrozen!

PHASE 2 — Full Fine-Tune (5 epochs)
Base LR: 2e-05  |  Head LR: 0.0002



Validation: 100%|██████████| 25/25 [00:15<00:00,  1.64it/s]


Epoch 1/5
  Train Loss : 1.3540
  Train F1   : 0.6290
  Val F1     : 0.5521
  ✓ Saved best Phase 2 model (val F1: 0.5521)


Validation: 100%|██████████| 25/25 [00:14<00:00,  1.69it/s]


Epoch 2/5
  Train Loss : 1.0165
  Train F1   : 0.7733
  Val F1     : 0.7522
  ✓ Saved best Phase 2 model (val F1: 0.7522)


...

Epoch 3/5
  Train Loss : 0.8694
  Train F1   : 0.8508
  Val F1     : 0.7801
  ✓ Saved best Phase 2 model (val F1: 0.7801)


Validation: 100%|██████████| 25/25 [00:14<00:00,  1.70it/s]


Epoch 4/5
  Train Loss : 0.7685
  Train F1   : 0.8981
  Val F1     : 0.7716


Validation: 100%|██████████| 25/25 [00:15<00:00,  1.65it/s]

Epoch 5/5
  Train Loss : 0.7173
  Train F1   : 0.9192
  Val F1     : 0.7730

Phase 2 done! Best val F1: 0.7801


In [26]:
# # ── CELL 17: TTA Inference Function ───────────────────
# # TTA still uses ASTFeatureExtractor (one call per crop, not the hot path).
# # Alternatively we use gpu_transform with augment=False for consistency.

# def predict_with_tta(model, audio, n_tta=N_TTA):
#     all_probs = []
#     for _ in range(n_tta):
#         cropped = crop_random(audio)
#         cropped = normalize(cropped)
#         # Use gpu_transform for consistency with training
#         waveform = torch.from_numpy(cropped).unsqueeze(0).to(DEVICE)  # (1, T)
#         with torch.no_grad():
#             input_values = gpu_transform(waveform, augment=False)     # (1, T_frames, 128)
#             outputs      = model(input_values=input_values)
#             probs        = torch.softmax(outputs.logits, dim=1)
#             all_probs.append(probs.cpu().numpy())

#     avg_probs = np.mean(all_probs, axis=0)
#     return np.argmax(avg_probs, axis=1)[0]


# print(f'TTA function ready! Using {N_TTA} crops per prediction.')

TTA function ready! Using 5 crops per prediction.


In [27]:
# # ── CELL 18: Generate Submission ──────────────────────
# # Load best model and predict on all test files

# model.load_state_dict(torch.load('/kaggle/working/best_model_phase2.pth'))
# model.eval()
# print(f'Best model loaded! Val F1 was: {best_p2_f1:.4f}')

# test_df   = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
# print(f'Test samples: {len(test_df)}')

# all_ids   = []
# all_preds = []

# for idx in tqdm(range(len(test_df)), desc=f'Inference (TTA x{N_TTA})'):
#     row   = test_df.iloc[idx]
#     path  = os.path.join(BASE_PATH, row['filename'])
#     audio = load_audio(path)
#     pred  = predict_with_tta(model, audio, n_tta=N_TTA)
#     all_ids.append(row['id'])
#     all_preds.append(id2label[pred])

# submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
# submission.to_csv('/kaggle/working/submission.csv', index=False)

# print(f'\nSubmission saved!')
# print(f'Total predictions : {len(submission)}')
# print(f'\nGenre distribution:')
# print(submission['genre'].value_counts())
# print(f'\nFirst 5 predictions:')
# print(submission.head())

Best model loaded! Val F1 was: 0.7801
Test samples: 3020


Inference (TTA x5): 100%|██████████| 3020/3020 [14:56<00:00,  3.37it/s]


Submission saved!
Total predictions : 3020

Genre distribution:
genre
pop          483
reggae       429
hiphop       344
metal        337
rock         318
jazz         305
classical    257
blues        230
disco        213
country      104
Name: count, dtype: int64

First 5 predictions:
   id    genre
0   1      pop
1   2     jazz
2   3      pop
3   4    metal
4   5  country


In [ ]:
# ── CELL 17: TTA Inference — Batched DataLoader (4 workers) ──
# BEFORE: sequential loop — one file → N_TTA single-sample GPU calls.
# AFTER : TestTTADataset flattens N_test × N_TTA into one flat Dataset.
#   • 4 workers load & crop waveforms in parallel (pure I/O + numpy)
#   • GPU sees full BATCH_SIZE batches → high utilisation
#   • probabilities are accumulated per file_idx and averaged at the end

class TestTTADataset(Dataset):
    """
    Each item is one (file, tta_crop) pair.
    Total length = len(test_df) × N_TTA.
    Returns (file_idx, waveform) so we can group probs by file after inference.
    """
    def __init__(self, test_df, base_path, n_tta=N_TTA):
        self.test_df   = test_df
        self.base_path = base_path
        self.n_tta     = n_tta
        self.n_files   = len(test_df)

    def __len__(self):
        return self.n_files * self.n_tta

    def __getitem__(self, idx):
        file_idx  = idx % self.n_files        # which test file
        row       = self.test_df.iloc[file_idx]
        path      = os.path.join(self.base_path, row['filename'])
        audio     = load_audio(path)           # torchaudio — fast
        cropped   = crop_random(audio)         # different crop each TTA pass
        cropped   = normalize(cropped)
        return file_idx, torch.from_numpy(cropped)  # (file_idx, T)


print(f'TestTTADataset class defined! ({N_TTA} crops × N_test files)')

In [ ]:
# ── CELL 18: Generate Submission ──────────────────────
model.load_state_dict(torch.load('/kaggle/working/best_model_phase2.pth'))
model.eval()
print(f'Best model loaded! Val F1 was: {best_p2_f1:.4f}')

test_df = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
n_files = len(test_df)
print(f'Test samples : {n_files}  |  TTA crops : {N_TTA}  |  Total items : {n_files * N_TTA}')

tta_dataset = TestTTADataset(test_df, BASE_PATH, n_tta=N_TTA)
tta_loader  = DataLoader(
    tta_dataset,
    batch_size         = BATCH_SIZE * 2,   # inference: no gradients → 2× batch fits
    shuffle            = False,            # keep file_idx ordering intact
    num_workers        = NUM_WORKERS,      # 4 workers load & crop in parallel
    pin_memory         = True,
    persistent_workers = True,
    prefetch_factor    = PREFETCH,
)

# Accumulate softmax probabilities per file: shape (n_files, NUM_LABELS)
prob_accum = torch.zeros(n_files, NUM_LABELS, dtype=torch.float32)

with torch.no_grad():
    for file_indices, waveforms in tqdm(tta_loader, desc=f'Inference (TTA ×{N_TTA})'):
        waveforms    = waveforms.to(DEVICE)
        input_values = gpu_transform(waveforms, augment=False)   # (B, 1024, 128)
        outputs      = model(input_values=input_values)
        probs        = torch.softmax(outputs.logits, dim=1).cpu() # (B, NUM_LABELS)

        # Scatter-add probs into the right file slot
        for i, fid in enumerate(file_indices):
            prob_accum[fid] += probs[i]

# Average over N_TTA crops and take argmax
prob_accum /= N_TTA
pred_ids    = prob_accum.argmax(dim=1).tolist()

all_ids   = test_df['id'].tolist()
all_preds = [id2label[p] for p in pred_ids]

submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f'\nSubmission saved!')
print(f'Total predictions : {len(submission)}')
print(f'\nGenre distribution:')
print(submission['genre'].value_counts())
print(f'\nFirst 5 predictions:')
print(submission.head())

In [28]:
# ── CELL 19: Score Summary ────────────────────────────
print('='*50)
print('TRAINING COMPLETE!')
print('='*50)
print(f'  Phase 1 best val F1 : {best_p1_f1:.4f}')
print(f'  Phase 2 best val F1 : {best_p2_f1:.4f}')
print(f'  TTA crops used      : {N_TTA}')
print(f'  Expected Kaggle F1  : {best_p2_f1 + 0.02:.4f} (approx)')
print('='*50)
print()
print('To improve further, try changing in Cell 4:')
print('  DURATION    = 25     (more audio context)')
print('  TRAIN_SIZE  = 6000   (more diversity)')
print('  EPOCHS_P2   = 7      (more fine-tuning)')
print('  LR_P2       = 1e-5   (safer updates)')
print('  N_TTA       = 7      (better inference)')

TRAINING COMPLETE!
  Phase 1 best val F1 : 0.5874
  Phase 2 best val F1 : 0.7801
  TTA crops used      : 5
  Expected Kaggle F1  : 0.8001 (approx)

To improve further, try changing in Cell 4:
  DURATION    = 25     (more audio context)
  TRAIN_SIZE  = 6000   (more diversity)
  EPOCHS_P2   = 7      (more fine-tuning)
  LR_P2       = 1e-5   (safer updates)
  N_TTA       = 7      (better inference)
